## Create extensions

An extension holds additional columns for another HATS Catalog. It can store different modalities (e.g. cutouts, spectra) in a separate location, and is only read when requested.

In [ ]:
import lsdb

import nested_pandas as npd
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc

from dask.distributed import Client
from hats_import import ExtensionArguments, pipeline_with_client
from hats.catalog.catalog_extension import ExtensionProperties
from hats_import.collection.arguments import CollectionArguments
from nested_pandas import TensorDtype
from nested_pandas.tensors import TensorExtensionArray
from upath import UPath

output_path = UPath("/sdf/data/rubin/cosmicdata/lsdb/genesis")

client = Client(n_workers=20, memory_limit="24GB", threads_per_worker=1)

### 1. Re-generate collection

Generate the `milestone2` collection and include the `object_id` column necessary to create the extensions. Follows method used in [generate_m2_dataset.ipynb](https://github.com/lincc-frameworks/genesis_deliverables/blob/main/phase1/milestone2/generate_m2_dataset.ipynb).

In [ ]:
ls_cutouts = lsdb.open_catalog("hf://datasets/hugging-science/mmu_legacysurvey_dr10_south_21", columns=["object_id", "ra", "dec", "image"])
desi_spec = lsdb.open_catalog("hf://datasets/UniverseTBD/mmu_desi_edr_sv3", columns=["object_id", "ra", "dec", "spectrum"])
mm_cat = ls_cutouts.crossmatch(desi_spec, suffix_method="overlapping_columns")
mm_cat.plot_pixels()

Use Doug's method to convert MMU nested image approach to Tensors:

In [ ]:
BANDS = ("des-g", "des-r", "des-i", "des-z")
IMAGE_SHAPE = (160, 160)
IMAGE_FIELDS = ("flux", "mask", "ivar")
SCALAR_FIELDS = ("psf_fwhm", "scale")

def _image_values(images: pa.Array, shape: tuple[int, int]) -> pa.Array:
    """Flatten list<list<T>> images into their values, checking they all have the given 2-d shape."""
    rows = pc.list_flatten(images)
    if pc.any(pc.not_equal(pc.list_value_length(images), shape[0])).as_py() or pc.any(
        pc.not_equal(pc.list_value_length(rows), shape[1])
    ).as_py():
        raise ValueError(f"Not all images have shape {shape}")
    return pc.list_flatten(rows)

def image_to_tensor_columns(
    nf: npd.NestedFrame,
    column: str = "image",
    bands=BANDS,
    image_shape=IMAGE_SHAPE,
    band_name=lambda band: band.removeprefix("des-"),
    drop: bool = True,
) -> npd.NestedFrame:
    """Replace a nested image column with one tensor column per (field, band).
    For every band in ``bands``, adds ``{column}_{field}_{band}`` tensor
    columns for flux, mask and ivar, and float columns for psf_fwhm and
    scale. Bands not in ``bands`` are ignored. The output columns and dtypes
    depend only on ``bands``, ``image_shape`` and the input schema, never on
    the data, so empty frames (e.g. dask/LSDB meta or empty partitions) give
    the same result schema. Rows that have no entry for a band get a missing
    tensor / NaN.
    """
    struct = nf[column].array.struct_array.combine_chunks()
    n = len(struct)
    flat_bands = pc.list_flatten(struct.field("band"))
    # Which top-level row each flat (per-band) entry belongs to
    parents = pc.list_parent_indices(struct.field("band"))
    new_columns = {}
    for band in bands:
        positions = pc.indices_nonzero(pc.equal(flat_bands, band))
        band_parents = pc.take(parents, positions)
        if len(pc.unique(band_parents)) != len(band_parents):
            raise ValueError(f"Some rows have band {band!r} more than once")
        # For each top-level row, index into this band's entries, or null when the row lacks the band
        take_idx = np.full(n, -1, dtype=np.int64)
        take_idx[band_parents.to_numpy()] = np.arange(len(positions))
        take_idx = pa.array(take_idx, mask=take_idx < 0)
        suffix = band_name(band)
        for field in IMAGE_FIELDS:
            images = pc.take(pc.list_flatten(struct.field(field)), positions)
            values = _image_values(images, image_shape)
            # Element type from the schema, so it is known even with no data
            dtype = TensorDtype(pa.fixed_shape_tensor(images.type.value_type.value_type, list(image_shape)))
            # Zero-copy: the flat values are already in C order
            band_tensors = pa.FixedSizeListArray.from_arrays(values, dtype.size)
            tensors = pc.take(band_tensors, take_idx)
            new_columns[f"{column}_{field}_{suffix}"] = pd.Series(
                TensorExtensionArray(tensors, dtype=dtype), index=nf.index, copy=False
            )
        for field in SCALAR_FIELDS:
            values = pc.take(pc.take(pc.list_flatten(struct.field(field)), positions), take_idx)
            new_columns[f"{column}_{field}_{suffix}"] = pd.Series(
                values, index=nf.index, dtype=pd.ArrowDtype(values.type)
            )
    out = nf.drop(columns=[column]) if drop else nf.copy()
    return out.assign(**new_columns)

In [ ]:
# Generate meta lazily (behavioral not computationally)
meta = image_to_tensor_columns(mm_cat.head(0), bands=("des-g", "des-r"))
milestone2_cat = mm_cat.map_partitions(image_to_tensor_columns, bands=("des-g", "des-r"), meta=meta)
milestone2_cat.write_catalog(output_path / "milestone2", overwrite=True)

Create a collection for this catalog, with margins:

In [ ]:
milestone2 = lsdb.open_catalog(output_path / "milestone2")
milestone2

In [ ]:
args = (
    CollectionArguments(
        output_artifact_name="milestone2",
        output_path=output_path,
        simple_progress_bar=True,
        resume=True,
    )
    .catalog(catalog_path=output_path / "milestone2" / milestone2_cat.name)
    .add_margin(margin_threshold=10, is_default=True)
)
pipeline_with_client(args, client)

### 2. Create cutouts/spectra extensions

An extension holds additional columns for another HATS Catalog. It can store different modalities (e.g. cutouts, spectra) in a separate location, and is only read when requested.

We will create an extension for the cutouts:

In [ ]:
args = ExtensionArguments(
    input_catalog_path=output_path / "milestone2",
    extension_columns=[f"image_{col}_{band}" for band in ("g", "r") for col in ["flux", "mask", "ivar"]],
    primary_column="object_id_mmu_legacysurvey_dr10_south_21",
    extension_alias="cutouts",
    extension_name="milestone2_cutouts",
    product_type_served="image",
    output_path=output_path,
    output_artifact_name="milestone2_with_cutouts",
    simple_progress_bar=True,
)
pipeline_with_client(args, client)

And another for the spectra:

In [ ]:
args = ExtensionArguments(
    input_catalog_path=output_path / "milestone2",
    extension_columns=["spectrum"],
    primary_column="object_id_mmu_legacysurvey_dr10_south_21",
    extension_alias="spectra",
    extension_name="milestone2_spectra",
    product_type_served="spectra",
    output_path=output_path,
    output_artifact_name="milestone2_with_spectra",
    simple_progress_bar=True,
    resume=True,
)
pipeline_with_client(args, client)

#### Workarounds

This pipeline only supports splitting on an extension at a time, and it creates the "core" collection every time. 

- When creating the first extension (for cutouts) the spectra will be included in the first "core" catalog. 
- When creating the second extension (for spectra) the cutouts will be included in the second "core" catalog.

We need to do some manual fudging:

In [ ]:
!ls $output_path

Let's move them to the `milestone2` collection directory:

In [ ]:
!rm -r $output_path/milestone2_with_cutouts/mmu_legacysurvey_dr10_south_21_x_mmu_desi_edr_sv3*
!rm -r $output_path/milestone2_with_cutouts/collection.properties
!mv $output_path/milestone2_with_cutouts/* $output_path/milestone2/
!rm -r $output_path/milestone2_with_cutouts

In [ ]:
!rm -r $output_path/milestone2_with_spectra/mmu_legacysurvey_dr10_south_21_x_mmu_desi_edr_sv3*
!rm -r $output_path/milestone2_with_spectra/collection.properties
!mv $output_path/milestone2_with_spectra/* $output_path/milestone2/
!rm -r $output_path/milestone2_with_spectra

In [ ]:
!ls $output_path/milestone2

And fix the `collection.properties`, to add the extensions:

In [ ]:
from hats.catalog.catalog_collection import CollectionProperties
props = CollectionProperties.read_from_file(output_path / "milestone2" / "collection.properties")
props = props.model_copy(update={"all_extensions": {"cutouts": "milestone2_cutouts.properties", "spectra": "milestone2_spectra.properties"}})
props.to_properties_file(output_path / "milestone2")

In [ ]:
!cat $output_path/milestone2_small/collection.properties

Workaround: I'll also use the extension pipeline to get a core catalog without "spectra" and "cutouts".

In [ ]:
args = ExtensionArguments(
    input_catalog_path=output_path / "milestone2",
    extension_columns=["spectrum"]+[f"image_{col}_{band}" for band in ("g", "r") for col in ["flux", "mask", "ivar"]],
    primary_column="object_id_mmu_legacysurvey_dr10_south_21",
    extension_alias="mixed",
    extension_name="mixed",
    product_type_served="mixed",
    output_path=output_path,
    output_artifact_name="milestone2_mixed",
    simple_progress_bar=True,
    resume=True,
)
pipeline_with_client(args, client)

In [ ]:
!ls $output_path/milestone2

In [ ]:
!mkdir $output_path/tmp
!mv $output_path/milestone2/mmu_legacysurvey_dr10_south_21_x_mmu_desi_edr_sv3* $output_path/tmp
!mv $output_path/milestone2_mixed/mmu_legacysurvey_dr10_south_21_x_mmu_desi_edr_sv3* $output_path/milestone2
!rm -r $output_path/tmp

The cutout extension is uploaded to the HF:

```
hf://datasets/LSDB/milestone2_cutouts
```

And the spectra extension is uploaded to shire @ UW:

```
https://data.lsdb.io/unstable/milestone2_spectra
```

So we can update the extension properties of each to point to the remote locations instead:

In [ ]:
props = ExtensionProperties.read_from_file(output_path / "milestone2" / "milestone2_cutouts.properties")
props = props.model_copy(update={"join_catalog": "hf://datasets/LSDB/milestone2_cutouts"})
props.to_properties_file(output_path / "milestone2")

In [ ]:
props = ExtensionProperties.read_from_file(output_path / "milestone2" / "milestone2_spectra.properties")
props = props.model_copy(update={"join_catalog": "https://data.lsdb.io/unstable/milestone2_spectra"})
props.to_properties_file(output_path / "milestone2")

In [ ]:
client.close()